Stencil system

Install the stencil_lib wheel using pip 

In [22]:
import subprocess, sys, glob, pathlib

# location of .whl file
_search_paths = [
    pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path("."),
    pathlib.Path("../../build/dist"),
]
_wheel = next(
    (str(w) for p in _search_paths for w in p.glob("stencil_lib-*.whl")),
    None
)
if _wheel is None:
    raise FileNotFoundError("stencil_lib wheel not found. Run 'inv build' or place the wheel alongside the notebook.")

# pip install the wheel
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       _wheel, "--force-reinstall"])


print(f"stencil_lib installed from: {_wheel}")


stencil_lib installed from: ..\..\build\dist\stencil_lib-0.1.0-py3-none-any.whl


Start using the stencil_lib library

In [23]:
# Start using the library
from stencil_lib import CipherConfig

In [24]:
class AESConfig(CipherConfig):
    algo = "aes"
    def __init__(self, key: bytes, mode: int, **mode_params):
        self.parameters = {
            "key": key,
            "mode": mode,
            "mode_params": mode_params
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        from Crypto.Cipher import AES
        cipher = AES.new(self.parameters["key"], self.parameters["mode"], **self.parameters["mode_params"])
        return cipher.encrypt(plaintext)
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        from Crypto.Cipher import AES
        cipher = AES.new(self.parameters["key"], self.parameters["mode"], **self.parameters["mode_params"])
        return cipher.decrypt(ciphertext)


In [25]:
class CaesarConfig(CipherConfig):
    algo = "caesar"
    def __init__(self, shift: int):
        self.parameters = {
            "shift": shift
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        shift = self.parameters["shift"]
        return bytes((b + shift) % 256 for b in plaintext)
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        shift = self.parameters["shift"]
        return bytes((b - shift) % 256 for b in ciphertext)


In [26]:
class VigenereConfig(CipherConfig):
    algo = "vigenere"
    def __init__(self, keyword: bytes):
        self.parameters = {
            "keyword": keyword
        }
    def encrypt(self, plaintext: bytes, *args, **kwargs) -> bytes:
        key = self.parameters["keyword"]
        key_len = len(key)
        return bytes((b + key[i % key_len]) % 256 for i, b in enumerate(plaintext))
    def decrypt(self, ciphertext: bytes, *args, **kwargs) -> bytes:
        key = self.parameters["keyword"]
        key_len = len(key)
        return bytes((b - key[i % key_len]) % 256 for i, b in enumerate(ciphertext))


# Stencil_system demo

The stencil library renders 3 APIs namely keygen, encrypt and decrypt

The usage of these APIs are demonstarated below.

In [ ]:
import random
import stencil_lib


grid_shape = stencil_lib.GridShape(n=2, shape=(12, 12))
in_byte_len = 16
plaintext = random.randbytes(in_byte_len)
num_partitions = 4
print(f"Plaintext : {' '.join(f'{b:02x}' for b in plaintext)}")

# cipher_cfg = AESConfig(key=random.randbytes(16), mode=1, iv=random.randbytes(16))
# cipher_cfg = CaesarConfig(shift=13)
cipher_cfg = VigenereConfig(keyword=b"KEY")

# --- New: create StencilConfig ---
stencil_cfg = stencil_lib.StencilConfig(
    total_bytes=in_byte_len,
    num_partitions=num_partitions,
    grid_shape=grid_shape,
)

Plaintext : a3 90 8f da 45 eb 72 bf aa 2d 0e f0 76 94 5a 7d


In [28]:
# Grid generation is optional. If Grid is none, 
# it is randomly generated internally inside encrypt API.
grid = stencil_lib.generate_random_grid(stencil_cfg.grid_shape)

rows, cols = grid_shape.shape
print("Initial Grid:")
for i in range(rows):
    row_bytes = [grid.get_value((i, j)) for j in range(cols)]
    hex_bytes = " ".join(f"{b:02x}" for b in row_bytes)
    print(f"  row {i:2d}: {hex_bytes}")

Initial Grid:
  row  0: eb 68 ae 19 d4 5b 4c 01 61 83 fc 4b
  row  1: a5 db d3 76 62 7a d6 be 89 81 7c 68
  row  2: fe f4 c7 2f 09 c4 3c 85 01 74 59 be
  row  3: 78 31 51 2e 19 e7 4e 93 0b 3d 20 af
  row  4: f3 d1 fe ba 2c 00 3b 4e 95 dc 75 d9
  row  5: eb 13 71 77 64 c7 25 f7 83 8c bc 35
  row  6: 39 ae 3e 95 f3 ee fc ec 82 49 1d 41
  row  7: f5 5d 93 a8 0f a9 b9 26 1b 03 aa e5
  row  8: 77 0d 1d 70 6a 89 e5 33 03 9f 7b d8
  row  9: e6 27 4c 44 46 71 cd b8 aa 58 2f ed
  row 10: db 19 05 2c 89 83 94 55 96 6c 8b 8b
  row 11: 13 74 29 9c 00 ba e2 6a e6 ad 2c cc


Keygen API

returns Secret key := stencil_lib.SecretKey type

In [29]:
secret_key = stencil_lib.keygen(
    stencil_cfg,
    cipher_cfg=cipher_cfg,
)

print("Secret Key")
print(f"  Cipher    : {secret_key.cipher_cfg.algo}")
print(f"  Partitions: {secret_key.partition_list}  ({len(secret_key.partition_list)} total, sum={sum(secret_key.partition_list)} bytes)")
print(f"  Stencils  : {len(secret_key.stencils)}")
for i, s in enumerate(secret_key.stencils):
    coords_str = ", ".join(f"({r},{c})" for r, c in s.coords)
    print(f"    [{i}] shape={s.shape!r}  len={s.len}  coords=[{coords_str}]")


Secret Key
  Cipher    : vigenere
  Partitions: [8, 4, 2, 2]  (4 total, sum=16 bytes)
  Stencils  : 4
    [0] shape='skewconnected'  len=8  coords=[(8,11), (8,10), (9,9), (9,8), (8,7), (9,6), (9,7), (10,8)]
    [1] shape='skewconnected'  len=4  coords=[(10,9), (10,10), (9,11), (9,10)]
    [2] shape='skewconnected'  len=2  coords=[(10,1), (9,1)]
    [3] shape='skewconnected'  len=2  coords=[(0,2), (1,3)]


Encrypt API 

returns obfuscated_grid := stencil_lib.Grid type

In [30]:
obfuscated_grid = stencil_lib.encrypt(plaintext, secret_key, stencil_cfg.grid_shape, grid)
print("\nWith Encryption API call, (Cipher text embedded) Obfuscated Grid is ready")



With Encryption API call, (Cipher text embedded) Obfuscated Grid is ready


Print Cipher text and Obfuscated grid for Demo purpose:

In [31]:
from IPython.display import display, HTML

# Just for Demo purpose
ciphertext = cipher_cfg.encrypt(plaintext)

print(f"Ciphertext: {' '.join(f'{b:02x}' for b in ciphertext)}")

PART_COLORS = ["#f0a500", "#4fc3f7", "#81c784", "#f06292", "#ce93d8", "#80cbc4"]

# --- Partitioned ciphertext ---
parts_lines = []
offset = 0
for i, length in enumerate(secret_key.partition_list):
    chunk = ciphertext[offset: offset + length]
    color = PART_COLORS[i % len(PART_COLORS)]
    hex_part = " ".join(f"{b:02x}" for b in chunk)
    parts_lines.append(
        f'  P{i} ({length:2d}B): <span style="color:{color};font-weight:bold">{hex_part}</span>'
    )
    offset += length

display(HTML(
    "<b>Ciphertext — by partition</b>"
    '<pre style="line-height:1.8">' + "\n".join(parts_lines) + "</pre>"
))

# --- Obfuscated grid: all bytes of a partition share its color; first byte underlined ---
coord_to_part  = {(r, c): i for i, s in enumerate(secret_key.stencils) for r, c in s.coords}
first_coords   = {s.coords[0] for s in secret_key.stencils}

rows_html = []
rows, cols = stencil_cfg.grid_shape.shape
for i in range(rows):
    cells = []
    for j in range(cols):
        b = obfuscated_grid.get_value((i, j))
        token = f"{b:02x}"
        if (i, j) in coord_to_part:
            part_idx = coord_to_part[(i, j)]
            color = PART_COLORS[part_idx % len(PART_COLORS)]
            underline = ";text-decoration:underline" if (i, j) in first_coords else ""
            token = f'<span style="color:{color};font-weight:bold{underline}">{token}</span>'
        cells.append(token)
    rows_html.append(f'  row {i:2d}: {' '.join(cells)}')

legend = "  ".join(
    f'<span style="color:{PART_COLORS[i % len(PART_COLORS)]};font-weight:bold">P{i}</span>'
    for i in range(len(secret_key.stencils))
)
display(HTML(
    f"<b>Obfuscated Grid</b> — {legend} (underlined = partition start)<br>"
    '<pre style="line-height:1.6">' + "\n".join(rows_html) + "</pre>"
))


Ciphertext: ee d5 e8 25 8a 44 bd 04 03 78 53 49 c1 d9 b3 c8


Decryprt API 

returns plaintext := bytes type

In [32]:
recovered = stencil_lib.decrypt(obfuscated_grid, secret_key)
assert recovered == plaintext

print(f"Decrypted Plaintext : {' '.join(f'{b:02x}' for b in recovered)}")

Decrypted Plaintext : a3 90 8f da 45 eb 72 bf aa 2d 0e f0 76 94 5a 7d
